# Уравнение диффузии-реакции Фишера

## Базовый уровень

Решаем следующее дифференциальное уравнение в частных производных

$$\frac{\partial u}{\partial t} = D \frac{\partial^2 u}{\partial x^2} + r u (1 - u), \quad x \in \left[0, 1\right], \quad D, r = \text{const}$$

с граничными условиями Неймана

$$\frac{\partial u}{\partial t} \left(0, t\right) = \frac{\partial u}{\partial t} \left(1, t\right) = 0.$$

Численное решение осуществим с помощью схемы FTCS и полунеявной схемы Кранка-Николсона.

Граничные условия дискретизируются с помощью конечных разностей

$$\frac{u_1 - u_0}{h} = 0,$$

$$\frac{u_n - u_{n - 1}}{h} = 0.$$

И сводятся к сносу с соседних ячеек:

$$u_0 = u_1, \quad u_n = u_{n - 1}.$$

Схема FTCS получается из следующей дискретизации:

$$\frac{u_i^{n+1} - u_i^n}{\Delta t} = D \frac{u_{i+1}^n - 2u_i^n + u_{i-1}^n}{h^2} + r\,u_i^n(1 - u_i^n).$$

Или в явном виде:

$$u_i^{n+1} = u_i^n + \alpha\,(u_{i+1}^n - 2u_i^n + u_{i-1}^n) + \Delta t\,r\,u_i^n(1 - u_i^n),$$
где $\alpha = D\frac{\Delta t}{h^2}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as anim

In [ ]:
def u0(x):
    return np.exp(-100*(x-0.5)**2)

In [ ]:
def ftcs(u, D, r, dt, h, N, timesteps):
    for i in range(timesteps):
        u[i+1, 0] = u[i, 1]
        u[i+1, N-1] = u[i, N-2]
        for j in range(1, N-1):
            u[i+1, j] = u[i, j] + D * dt / h ** 2 * (u[i, j+1] - 2 * u[i, j] + u[i, j-1]) + dt * r * u[i, j] * (1 - u[i, j])
    return u

In [ ]:
N = 100
timesteps = 2100
x = np.linspace(0.0, 1.0, N)
h = x[1] - x[0]
T = 10.0
dt = T / timesteps

D = 0.01
r = 1.0

# initial conditions
u = np.zeros((timesteps+1, N))
u[0, :] = np.exp(-100 * (x - 0.5) ** 2)

u_cn = np.zeros((timesteps+1, N))
u_cn[0, :] = u0(x)

In [ ]:
alpha = D * dt / h ** 2
print('Число Куранта =', alpha)

In [ ]:
u = ftcs(u, D, r, dt, h, N, timesteps)

In [ ]:
fig, ax = plt.subplots()
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.1)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('u(t, x)')
ax.set_title('Решение уравнения Фишера (FTCS),\nчисло Куранта = {}'.format(alpha))
ax.grid(True)

line, = ax.plot(x, u[0, :], c='blue', alpha=0.5)

def animate(i, x, u):
    line.set_data(x, u[i, :])
    return line

ani = anim.FuncAnimation(fig, animate, frames=timesteps // 50, interval=100, fargs=(x, u[::50],))

from IPython.display import HTML
HTML(ani.to_jshtml())

Решение уравнения стремится к $u \equiv 1$: популяция распространяется по всей области $x \in \left[0, 1\right]$; можно отметить, что сначала популяция в точке $x = 0.5$ уменьшается (из-за влияния диффузионного члена в уравнении), а затем увеличивается и доходит до 1.

Полунеявная схема Кранка-Николсона основана на усреднении по времени диффузионного члена:
$$\frac{u_i^{n+1} - u_i^n}{\tau} = D\,\frac{1}{2}\left[\frac{u_{i+1}^{n+1} - 2u_i^{n+1} + u_{i-1}^{n+1}}{h^2} + \frac{u_{i+1}^{n} - 2u_i^{n} + u_{i-1}^{n}}{h^2}\right] + r\,u_i^n(1 - u_i^n).$$

Переносим неизвестные $u^{n+1}$ влево:
$$
-\frac{\alpha}{2} u_{i-1}^{n+1}
+ (1 + \alpha) u_i^{n+1}
- \frac{\alpha}{2} u_{i+1}^{n+1}
= u_i^n + \frac{\alpha}{2}(u_{i+1}^n - 2u_i^n + u_{i-1}^n)
+ \Delta t\,r\,u_i^n(1 - u_i^n),
$$
где $\alpha = D\frac{\tau}{h^2}$.

Таким образом, на каждом шаге $n \to n+1$ решается трёхдиагональная система для $u^{n+1}$.

In [ ]:

def thomas_algorithm(a, b, c, d):
    n = len(d)
    c_prime = np.zeros(n)
    d_prime = np.zeros(n)
    x = np.zeros(n)

    # Прямой ход (forward elimination)
    c_prime[0] = c[0] / b[0]
    d_prime[0] = d[0] / b[0]

    for i in range(1, n):
        denom = b[i] - a[i] * c_prime[i-1]
        if i < n - 1:
            c_prime[i] = c[i] / denom
        d_prime[i] = (d[i] - a[i] * d_prime[i-1]) / denom

    # Обратный ход (back substitution)
    x[-1] = d_prime[-1]
    for i in range(n - 2, -1, -1):
        x[i] = d_prime[i] - c_prime[i] * x[i+1]

    return x

def crank_nicolson(u, D, r, dx, dt, N, timesteps):
    alpha = D * dt / dx ** 2
    
    lower = np.zeros(N)      # нижняя диагональ
    diag = np.zeros(N)       # главная диагональ
    upper = np.zeros(N)      # верхняя диагональ
    
    # Граничное условие Неймана на левом конце
    diag[0] = 1 + alpha
    upper[0] = -alpha
    
    # Внутренние точки
    for i in range(1, N-1):
        lower[i] = -alpha / 2
        diag[i] = 1 + alpha
        upper[i] = -alpha / 2
    
    # Граничное условие Неймана на правом конце
    lower[-1] = -alpha
    diag[-1] = 1 + alpha
    
    for n in range(timesteps):
        rhs = np.zeros(N)
        rhs[0] = (1 - alpha) * u[n, 0] + alpha * u[n, 1] + dt * r * u[n, 0] * (1 - u[n, 0])
        
        # Внутренние точки
        for i in range(1, N-1):
            rhs[i] = (alpha/2) * u[n, i-1] + (1 - alpha) * u[n, i] + (alpha/2) * u[n, i+1] + dt * r * u[n, i] * (1 - u[n, i])
        
        # Граничное условие Неймана на правом конце
        rhs[-1] = alpha * u[n, -2] + (1 - alpha) * u[n, -1] + dt * r * u[n, -1] * (1 - u[n, -1])
        
        # Решение трехдиагональной системы методом прогонки
        u[n+1, :] = thomas_algorithm(lower, diag, upper, rhs)

    return u

In [ ]:
u_cn = crank_nicolson(u_cn, D, r, h, dt, N, timesteps)

In [ ]:
fig, ax = plt.subplots()
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.1)
ax.set_aspect('equal')
ax.grid(True)
ax.set_xlabel('x')
ax.set_ylabel('u(t, x)')
ax.set_title('Решение уравнения Фишера\n(полуявный Кранк-Николсон),\nчисло Куранта = {}'.format(alpha))

line, = ax.plot(x, u_cn[0, :], c='red', alpha=0.5)

ani = anim.FuncAnimation(fig, animate, frames=timesteps // 50, interval=100, fargs=(x, u_cn[::50],))

from IPython.display import HTML
HTML(ani.to_jshtml())

### Анализ влияния шага $\Delta t$ на устойчивость FTCS

In [ ]:
N = 100
timesteps = 900
alpha = 0.51
x = np.linspace(0.0, 1.0, N)
h = x[1] - x[0]
dt = alpha * h ** 2 / D
T = dt * timesteps

u = np.zeros((timesteps + 1, N))
u[0, :] = u0(x)

In [ ]:
u = ftcs(u, D, r, dt, h, N, timesteps)

In [ ]:
fig, ax = plt.subplots()
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.1)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('u(t, x)')
ax.grid(True)
ax.set_title('Решение уравнения Фишера (FTCS),\nчисло Куранта = {}'.format(alpha))

line, = ax.plot(x, u[0, :], c='blue', alpha=0.5)

def animate(i, x, u):
    line.set_data(x, u[i, :])
    return line

ani = anim.FuncAnimation(fig, animate, frames=timesteps // 25, interval=100, fargs=(x, u[::25],))

from IPython.display import HTML
HTML(ani.to_jshtml())

In [ ]:
timesteps = 2000
alpha = 0.5

dt = alpha * h ** 2 / D
T = dt * timesteps

u = np.zeros((timesteps+1, N))
u[0, :] = u0(x)

In [ ]:
u = ftcs(u, D, r, dt, h, N, timesteps)

In [ ]:
fig, ax = plt.subplots()
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.1)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('u(t, x)')
ax.grid(True)
ax.set_title('Решение уравнения Фишера (FTCS),\nчисло Куранта = {}'.format(alpha))

line, = ax.plot(x, u[0, :], c='blue', alpha=0.5)

def animate(i, x, u):
    line.set_data(x, u[i, :])
    return line

ani = anim.FuncAnimation(fig, animate, frames=timesteps // 100, interval=100, fargs=(x, u[::100],))

from IPython.display import HTML
HTML(ani.to_jshtml())

Схема FTCS неустойчива при $\Delta t \gt \frac{1}{2} \frac{h^2}{D}$.

## Продвинутый уровень

Реализуем полный нелинейный метод Кранка-Николсона, а именно на каждом шаге по времени будем решать методом Ньютона-Рафсона следующую нелинейную систему алгебраических уравнений:

\begin{equation*}
u^{n+1}_i - \frac{D \Delta t}{2 h^2} \left(u^{n+1}_{i-1} - 2 u^{n+1}_i + u^{n+1}_{i+1}\right) - \frac{r \Delta t}{2} u^{n+1}_i \left(1 - u^{n+1}_i\right)
=
\frac{D \Delta t}{2 h^2} \left(u^n_{i-1} - 2 u^n_i + u^n_{i+1}\right) + \frac{r \Delta t}{2} u^n_i \left(1 - u^n_i\right).
\end{equation*}

In [ ]:
def build_A(N, dx):
    A = np.zeros((N, N))
    for i in range(1, N-1):
        A[i, i-1] = 1.0
        A[i, i]   = -2.0
        A[i, i+1] = 1.0
    A[0,0] = -2.0; A[0,1] = 2.0
    A[-1,-1] = -2.0; A[-1,-2] = 2.0
    return A / (dx*dx)

def f(u):
    return u*(1-u)

def fprime(u):
    return 1.0-2.0*u

def crank_nicolson_step(u_n, A, D, r, dt, tol=1e-8, maxiter=30):
    N = u_n.size
    I = np.eye(N)
    rhs_const = u_n + dt*(D/2.0*A.dot(u_n) + r/2.0*f(u_n))
    u = u_n + dt*(D*A.dot(u_n) + r*f(u_n))
    total_iters = 0
    for k in range(maxiter):
        total_iters += 1
        Gu = u - rhs_const - dt*(D/2.0*A.dot(u) + r/2.0*f(u))
        res = np.linalg.norm(Gu)
        if res < tol:
            return u, total_iters, res
        J = I - dt*(D/2.0*A + r/2.0*np.diag(fprime(u)))
        try:
            delta = np.linalg.solve(J, -Gu)
        except np.linalg.LinAlgError:
            delta = -Gu
        u = u + delta
    return u, total_iters, np.linalg.norm(Gu)

In [ ]:
def nonlinear_crank_nicolson(u, A, D, r, dt, timesteps):
    total_iter = 0
    for i in range(timesteps):
        u[i+1], iters, diff = crank_nicolson_step(u[i], A, D, r, dt)
        total_iter += iters
    return u, total_iter, diff

Также реализуем метод линий. Дискретизируем исходную задачу по пространству $\mathbf u = \left(u_1, u_2, ..., u_N\right)^T$, а по времени будем решать обыкновенное дифференциальное уравнение методом Рунге-Кутты 4-го порядка:

\begin{equation*}
\frac{d \mathbf u}{d t} = D A \mathbf u + r \mathbf F(\mathbf u),
\end{equation*}

где $A$ -- матрица, аппроксимирующая вторую производную по пространству, а $F_i(\mathbf u) = u_i (1 - u_i)$.

In [ ]:
def rhs_mol(u, A, D, r):
    return D*A.dot(u) + r*f(u)

def rk4_step(u, A, D, r, dt):
    k1 = rhs_mol(u, A, D, r)
    k2 = rhs_mol(u + 0.5*dt*k1, A, D, r)
    k3 = rhs_mol(u + 0.5*dt*k2, A, D, r)
    k4 = rhs_mol(u + dt*k3, A, D, r)
    return u + dt*(k1 + 2*k2 + 2*k3 + k4)/6.0

def mol(u, A, D, r, dt, timesteps):
    for i in range(timesteps):
        u[i+1] = rk4_step(u[i], A, D, r, dt)
    return u

In [ ]:
N = 100
x = np.linspace(0.0, 1.0, N)
dx = x[1] - x[0]

timesteps = 2000
T = 10.0
dt = T / timesteps

A = build_A(N, dx)

D = 0.01
r = 1.0

u_ftcs = np.zeros((timesteps + 1, N))
u_ftcs[0] = u0(x)
u_cn = np.zeros((timesteps + 1, N))
u_cn[0] = u0(x)
u_cn_nonlinear = np.zeros((timesteps + 1, N))
u_cn_nonlinear[0] = u0(x)
u_mol = np.zeros((timesteps + 1, N))
u_mol[0] = u0(x)

In [ ]:
CFL = D * dt / dx
print('CFL = {}'.format(CFL))

In [ ]:
u_ftcs = ftcs(u, D, r, dt, dx, N, timesteps)
u_cn = crank_nicolson(u, D, r, dx, dt, N, timesteps)
u_cn_nonlinear, _, _ = nonlinear_crank_nicolson(u_cn_nonlinear, A, D, r, dt, timesteps)
u_mol = mol(u, A, D, r, dt, timesteps)


In [ ]:
fig, ax = plt.subplots()
ax.set_xlabel('x')
ax.set_ylabel('u(t, x)')
ax.grid(True)
ax.set_title('Решение уравнения Фишера (FTCS),\nчисло Куранта = {}'.format(CFL))

line1, = ax.plot(x, u_ftcs[0, :], alpha=0.5)
line2, = ax.plot(x, u_cn[0, :], alpha=0.5)
line3, = ax.plot(x, u_cn_nonlinear[0, :], alpha=0.5)
line4, = ax.plot(x, u_mol[0, :], alpha=0.5)

ax.legend(['FTCS', 'IMEX Crank-Nicolson', 'Non-linear Crank-Nicolson', 'Method of lines'])

def animate(i, x, u_ftcs, u_cn, u_cn_nonlinear, u_mol):
    line1.set_data(x, u_ftcs[i, :])
    line2.set_data(x, u_cn[i, :])
    line3.set_data(x, u_cn_nonlinear[i, :])
    line4.set_data(x, u_mol[i, :])
    return line1, line2, line3, line4,

ani = anim.FuncAnimation(fig, animate, frames=timesteps // 100, interval=100, fargs=(x, u_ftcs[::100], u_cn[::100], u_cn_nonlinear[::100], u_mol[::100],))

from IPython.display import HTML
HTML(ani.to_jshtml())

Сравним количество итераций и точность для разных $\Delta t$.

In [ ]:
N = 100
x = np.linspace(0.0, 1.0, N)
dx = x[1] - x[0]
T = 5.0

D = 0.01
r = 1.0
dt_array = np.array([1e-3, 5e-3, 1e-2, 5e-2, 1e-1, 5e-1])
timesteps_array = np.int32(T / dt_array)
u_t = []
for i in range(len(dt_array)):
    u_t.append(np.zeros((timesteps_array[i]+1, N)))
    u_t[i][0] = u0(x)
iter_array = []
err_array = []

dt_ref = 5e-4
nsteps_ref = int(np.ceil(T/dt_ref))
u_ref = u0(x)
A = build_A(N, dx)
for n in range(nsteps_ref):
    u_ref = rk4_step(u_ref, A, D, r, dt_ref)


In [ ]:
for i in range(len(dt_array)):
    u_t[i], iter, _ = nonlinear_crank_nicolson(u_t[i], A, D, r, dt_array[i], timesteps_array[i])
    err = np.linalg.norm(u_t[i][-1] - u_ref) * np.sqrt(1.0/N)
    print('dt = {}, число шагов по времени = {}, общее число итераций метода Ньютона-Рафсона = {}, среднее число итераций на шаг по времени = {}, ошибка = {}'.format(dt_array[i], timesteps_array[i], iter, iter / timesteps_array[i], err))
    iter_array.append(iter)
    err_array.append(err)

При увеличении $\Delta t$ уменьшается общее число итераций метода Ньютона-Рафсона (поскольку уменьшается число шагов по времени), но увеличивается ошибка (норма разности с референсным решением, полученным методом линий) и среднее число итераций алгоритма Ньютона-Рафсона на шаг по времени.

Построим семейство решений для $D \in [0.001, 0.1]$ и $r \in [0.5, 2]$.

In [ ]:
timesteps = 1000
T = 0.5
dt = T / timesteps
D_array = np.linspace(0.001, 0.1, 3)
r_array = np.linspace(0.5, 2.0, 3)
u_Dr = np.zeros((len(D_array), len(r_array), timesteps+1, N))
for i in range(len(D_array)):
    for j in range(len(r_array)):
        u_Dr[i, j, 0] = u0(x)
        u_Dr[i, j], _,  _ = nonlinear_crank_nicolson(u_Dr[i, j], A, D_array[i], r_array[j], dt, timesteps)

In [ ]:
for i in range(len(D_array)):
    for j in range(len(r_array)):
        plt.plot(x, u_Dr[i, j, -1], label='D = {}, r = {}'.format(D_array[i], r_array[j]))
plt.legend()
plt.xlabel('x')
plt.ylabel('u(t, x)')
plt.grid(True)
plt.title('Решения уравнения Фишера при различных параметрах D, r')
plt.show()

Из графиков видно, что коэффициент $r$ отвечает за скорость роста популяции, а $D$ за скорость распространения популяции по пространству (диффузию).